# 02 — Data Preprocessing & Leakage Prevention

**Project:** Customer Complaint Root-Cause Classifier  
**Module:** Data Preprocessing & Validation  

### Objectives:
1. Load raw 100,000 complaints dataset from `data/raw/customer_complaints.csv`.
2. Standardize and clean categorical and text fields (whitespace, datetimes, missing values).
3. Analyze narrative duplication and quantify the data leakage risk in naive random splitting.
4. Implement MD5 narrative hashing (`generate_narrative_hash`) for deterministic group identification.
5. Verify `GroupShuffleSplit` ensures **zero leakage** between train and test sets.
6. Validate memory and feature footprints for low-spec (8 GB RAM) production constraints.

## 1. Setup and Environment

In [ ]:
import sys
import os
from pathlib import Path

project_root = Path(os.path.abspath('')).resolve()
if project_root.name == 'notebooks':
    project_root = project_root.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.data.load_data import load_complaints
from src.data.clean_data import clean_complaints
from src.utils.helpers import generate_narrative_hash, get_memory_usage
from sklearn.model_selection import GroupShuffleSplit

print(f'Initial RSS Memory Usage: {get_memory_usage():.1f} MB')

## 2. Ingesting Raw Complaint Records

In [ ]:
df_raw = load_complaints()
print(f'Raw dataset shape: {df_raw.shape}')
df_raw.head(3)

## 3. Data Cleaning and Schema Standardization

In [ ]:
df_clean = clean_complaints(df_raw)
print(f'Cleaned dataset shape: {df_clean.shape}')
print('\nMissing values per column:')
print(df_clean.isnull().sum())
print('\nData types:')
print(df_clean.dtypes)

## 4. Narrative Duplication & Leakage Analysis

In [ ]:
df_clean['narrative_hash'] = df_clean['complaint_narrative'].apply(generate_narrative_hash)
unique_narratives = df_clean['narrative_hash'].nunique()
total_complaints = len(df_clean)
duplicate_narratives = total_complaints - unique_narratives

print(f'Total complaints: {total_complaints:,}')
print(f'Unique narrative hashes: {unique_narratives:,}')
print(f'Duplicate narratives: {duplicate_narratives:,} ({duplicate_narratives / total_complaints * 100:.2f}%)')

## 5. Group-Aware Train/Test Splitting Validation

In [ ]:
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(df_clean, groups=df_clean['narrative_hash']))

train_df = df_clean.iloc[train_idx]
test_df = df_clean.iloc[test_idx]

train_hashes = set(train_df['narrative_hash'])
test_hashes = set(test_df['narrative_hash'])
overlap = train_hashes.intersection(test_hashes)

print(f'Train set: {len(train_df):,} rows ({len(train_hashes):,} unique narratives)')
print(f'Test set: {len(test_df):,} rows ({len(test_hashes):,} unique narratives)')
print(f'Overlap between Train and Test: {len(overlap)} narratives (ZERO LEAKAGE CONFIRMED)')
assert len(overlap) == 0, 'Data leakage detected!'

## 6. Feature Distributions Post-Cleaning

In [ ]:
train_df_copy = train_df.copy()
train_df_copy['char_count'] = train_df_copy['complaint_narrative'].str.len()
train_df_copy['word_count'] = train_df_copy['complaint_narrative'].str.split().str.len()

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
sns.histplot(train_df_copy['char_count'], bins=35, ax=axes[0], color='royalblue', kde=True)
axes[0].set_title('Narrative Character Length Distribution')
axes[0].set_xlabel('Character Count')

sns.histplot(train_df_copy['word_count'], bins=35, ax=axes[1], color='darkorange', kde=True)
axes[1].set_title('Narrative Word Count Distribution')
axes[1].set_xlabel('Word Count')
plt.tight_layout()
plt.show()

print(f'Final Process RSS Memory Usage: {get_memory_usage():.1f} MB')